In [1]:

import pandas as pd
from engine import load_env, create_engine_from_env
from models.manufacture_module import get_manufacturer_df, get_active_unique_manufacturers, get_manufacturer_bulletins_json, print_bulletin_details, convert_bulletin_to_df, save_vehicle_models_to_csv, batch_save_manufacturer_models, search_models_by_description
 

In [2]:

import json
from typing import Union, Any

def parse_json_string(json_string: str) -> Union[dict, list]:
    """
    Parse a JSON string into a Python object (dict or list).

    Args:
        json_string (str): A valid JSON string.

    Returns:
        dict or list: Parsed JSON object.

    Raises:
        ValueError: If the input is not valid JSON.
        TypeError: If input is not a string.
    """
    if not isinstance(json_string, str):
        raise TypeError("Input must be a JSON string")

    try:
        return json.loads(json_string)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON string: {exc}") from exc
 

In [3]:

# Loaded environment variables and created database engine
load_env()
engine = create_engine_from_env()


python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 8


In [4]:

# # # Batch save vehicle models for specified manufacturers
# manufactures = ["Hyundai","Honda","Kia","Mazda","Genesis", "Mitsubishi", "Volkswagen"]

# # batch_save_manufacturer_models(engine, manufactures)


In [5]:

model_number_db = pd.read_csv("db/db_vehicle_models.csv")


In [6]:
model_number_db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Description   279 non-null    object
 1   Drivetrain    279 non-null    object
 2   Manufacturer  279 non-null    object
 3   ModelName     279 non-null    object
 4   ModelNumber   279 non-null    object
 5   ModelYear     279 non-null    int64 
 6   Package       279 non-null    int64 
 7   PassDoors     279 non-null    int64 
 8   TrimName      270 non-null    object
 9   engine_type   116 non-null    object
dtypes: int64(3), object(7)
memory usage: 21.9+ KB


In [7]:

model_number_db[model_number_db["engine_type"].isnull()]


,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,TrimName,engine_type
9,Advanced Awd,ALL_WHEEL_DRIVE,GENESIS,GV60,V6EW5ZE227AD,2024,444967,4,Advanced,NaN
10,Performance Awd,ALL_WHEEL_DRIVE,GENESIS,GV60,V6EW5ZE127AR,2024,444968,4,Performance,NaN
11,Performance Awd W/navy Interior,ALL_WHEEL_DRIVE,GENESIS,GV60,V6EW5ZE127PF,2024,444969,4,Performance,NaN
35,Advanced Awd,ALL_WHEEL_DRIVE,GENESIS,GV60,V6EW5ZE227AD,2025,461704,4,Advanced,NaN
36,Performance Awd,ALL_WHEEL_DRIVE,GENESIS,GV60,V6EW5ZE127AR,2025,461705,4,Performance,NaN
...,...,...,...,...,...,...,...,...,...,...
274,Essential Ivt W/two-Tone,FRONT_WHEEL_DRIVE,HYUNDAI,Venue,VNCW5V1FESTT,2026,480301,4,Essential,NaN
275,Preferred Ivt,FRONT_WHEEL_DRIVE,HYUNDAI,Venue,VNCW5V1FPR00,2026,480302,4,Preferred,NaN
276,Preferred Ivt W/two-Tone,FRONT_WHEEL_DRIVE,HYUNDAI,Venue,VNCW5V1FPRTT,2026,480303,4,Preferred,NaN
277,Ultimate Ivt W/black Interior,FRONT_WHEEL_DRIVE,HYUNDAI,Venue,VNCW5V1FUL00,2026,480304,4,Ultimate,NaN


In [8]:

model_number_db["engine_type"].unique()


array(['electric', '2.5t', '3.3t', '3.5t', 'e-sc', nan, 'hybrid'],
      dtype=object)

In [9]:
make_df_genesis = model_number_db[model_number_db["Manufacturer"].str.contains("Genesis", case=False)]
make_df_hyundai = model_number_db[model_number_db["Manufacturer"].str.contains("Hyundai", case=False)]

In [10]:

make_df_genesis[make_df_genesis["TrimName"].isna()]


,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,TrimName,engine_type
0,Awd,ALL_WHEEL_DRIVE,GENESIS,Electrified G80,G8ES4ZE1GP00,2024,450310,4,NaN,electric
61,Awd *ltd Avail*,ALL_WHEEL_DRIVE,GENESIS,GV60,V6EW5ZE2GW00,2026,481701,4,NaN,NaN


In [11]:

# make_df_hyundai[
    
#     (make_df_hyundai["ModelYear"]==2024)
#     # (1==1)
#     &(make_df_hyundai["Description"].str.contains("Es", case=False, na=False))
#     # # & (make_df_hyundai["Description"].str.contains("Cross", case=False, na=False))
#     # # & (make_df_hyundai["Description"].str.contains("noir", case=False, na=False))
#     # & (make_df_hyundai["Package"]=="KE00")
#     ]
 

In [31]:

# make = "Hyundai"
# year = 2024
# keywords = ['Elantra', "N"]

make = "Genesis"
year = 2024
keywords = ['G90']


search_models_by_description(make, year, keywords)


[SEARCH_DEBUG] After keyword 'G90': 1 records


""


In [32]:
year = 2024
Manufacturer = "Genesis"
ModelName = 'G90'

model_number_db[
    (1==1)
    & (model_number_db["ModelYear"] == year)
    & (model_number_db["Manufacturer"].str.contains(Manufacturer, case=False))
    & (model_number_db["ModelName"].str.contains(ModelName, case=False, na=False))
    ][["ModelYear", "ModelNumber", "ModelName", "TrimName", "Package", "engine_type","Manufacturer"]]

 

,ModelYear,ModelNumber,ModelName,TrimName,Package,engine_type,Manufacturer
8,2024,G9CS4K3BXXPS,G90,e-SC Prestige,450256,e-sc,GENESIS


I want us to make a change to the model look up.

We are getting miss-match in some instances where data in the csv db is not standardized to meet the expected keyword standard in the translator targets. This means that while the translator is translating, some keywords in teh csv db are still in the previous form. for instance, the translator is set to take ult -> ultimate. however some db records have utl which make the search to fail.  

The solution that I was thinking, add a step on the db data pull utility.

The process that i am thinking is as follows:

1. Pull the db,

2. Clean up the db. 

3. Standardize keywords: based on translator_keywors







I want to add an edge case that we need to handle:

There are some vehicles that have 2 model numbers, the old model and the new model. The characteristic, they are ht esame mafucture, year and the description is the same. The current logic states that if the search_model utility gets two records, it should flag as ambigous, but for this one, I wna us to add another condition, that if they are more than one option, but the year and description are the same, do the following:

Duplicate the rows for that search key
and create a records for each of the model numbers.

Fore example:

If we pass in :
make = "Hyundai"
year = 2024
keywords = ['elantra', 'ess']

We'll get the following two records:

380	Hyundai	2024	ELCS4V2BES00	Elantra Essential IVT			
515	Hyundai	2024	EL74IF20A100	Elantra Essential IVT			


At the end we'll have a list of parts records with ELCS4V2BES00 and another one with EL74IF20A100 as the model number.

I want this feature to be given a flag that will enable or disable it inside the OEM config. This way we can control it better. 

Give me a plan on how to do empliment this.


There are some keywords that I want us to ignore when it comes to matching. 

For instance, 




My 

In [11]:

from pathlib import Path
from semantic.translator import load_oem_translator
configs_dir = Path(r"C:\Users\paxm\OneDrive - PBS SYSTEMS\Desktop\Office\Projects\OEM Accessory project\OEM_Accessories_v1\accy_v2\model_lookup\configs")
translator = load_oem_translator("Hyundai", str(configs_dir))
print(f"Loaded {len(translator)} translator entries")
print("Sample entries:", dict(list(translator.items())[:5]))


Loaded 21 translator entries
Sample entries: {'pref': 'preferred', 'ess': 'essential', 'calli': 'calligraphy', 'lux': 'luxury', 'ult': 'ultimate'}


In [12]:
# Trace a specific description
from models.manufacture_module import _standardize_description, _clean_description_punctuation
test = "Ioniq 5 Pref calli AWD ult ed Long Range with lux pkg"
cleaned = _clean_description_punctuation(test)
standardized = _standardize_description(cleaned, translator)
print(f"Original:      {test}")
print(f"After clean:   {cleaned}")
print(f"After standard: {standardized}")

Original:      Ioniq 5 Pref calli AWD ult ed Long Range with lux pkg
After clean:   Ioniq 5 Pref calli AWD ult ed Long Range with lux pkg
After standard: Ioniq 5 Preferred Calligraphy Awd Ultimate Edition Long Range With Luxury Package
